# Modeliranje - LightGBM + Prophet features

Metrika: RMSLE (Root Mean Squared Log Error)

Posto je target vec u log prostoru (`sales_log = log1p(sales)`), RMSE na predikcijama = RMSLE na originalnim vrijednostima.

Output: `models/lgbm_prophet.pkl`

In [ ]:
import sys, subprocess
print('Python:', sys.executable)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'lightgbm', 'pyarrow', '-q'], check=True)
print('Instalacija gotova.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
import pickle
import os
import warnings

warnings.filterwarnings('ignore')

PROCESSED = '../data/processed/'
MODELS    = '../models/'
os.makedirs(MODELS, exist_ok=True)

## 1. Ucitavanje podataka

In [ ]:
train_df = pd.read_parquet(PROCESSED + 'train_features.parquet')
val_df   = pd.read_parquet(PROCESSED + 'val_features.parquet')

print(f'Train: {train_df.shape}')
print(f'Val:   {val_df.shape}')

## 2. Definisanje features i targeta

In [ ]:
FEATURES = [
    # store info
    'store_nbr', 'family_enc', 'type_enc', 'city_enc', 'state_enc', 'cluster',
    # promocija
    'onpromotion',
    # transakcije (moze imati NaN - LightGBM to hendla nativno)
    'transactions',
    # nafta
    'oil_price',
    # kalendar
    'year', 'month', 'weekofyear', 'dayofweek', 'dayofmonth',
    'is_weekend', 'is_payday',
    # praznici i eventi
    'is_national_holiday', 'is_local_holiday',
    'days_after_earthquake',
    # lag i rolling
    'lag_7', 'lag_14', 'lag_28', 'lag_56',
    'roll_mean_7', 'roll_mean_14', 'roll_mean_28',
    'roll_std_7',  'roll_std_14',  'roll_std_28',
    # prophet komponente
    'prophet_trend', 'prophet_weekly', 'prophet_yearly', 'prophet_yhat',
]

TARGET = 'sales_log'

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_val = val_df[FEATURES]
y_val = val_df[TARGET]

print(f'Features: {len(FEATURES)}')
print(f'X_train: {X_train.shape}, X_val: {X_val.shape}')

## 3. Treniranje LightGBM modela

In [ ]:
params = {
    'objective':        'regression',
    'metric':           'rmse',
    'learning_rate':    0.05,
    'num_leaves':       127,
    'max_depth':        -1,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':     5,
    'reg_alpha':        0.1,
    'reg_lambda':       0.1,
    'n_jobs':           -1,
    'verbose':          -1,
    'seed':             42,
}

dtrain = lgb.Dataset(X_train, label=y_train)
dval   = lgb.Dataset(X_val,   label=y_val, reference=dtrain)

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=100),
]

model = lgb.train(
    params,
    dtrain,
    num_boost_round=2000,
    valid_sets=[dtrain, dval],
    valid_names=['train', 'val'],
    callbacks=callbacks,
)

print(f'\nBest iteration: {model.best_iteration}')

## 4. Evaluacija na validacijskom setu

In [ ]:
val_pred_log = model.predict(X_val, num_iteration=model.best_iteration)

val_pred   = np.expm1(val_pred_log).clip(min=0)
val_actual = np.expm1(y_val)

rmsle = np.sqrt(np.mean((np.log1p(val_pred) - np.log1p(val_actual))**2))
mae   = np.mean(np.abs(val_pred - val_actual))
rmse  = np.sqrt(np.mean((val_pred - val_actual)**2))

print(f'Validacijski set (2017-08-01 do 2017-08-15):')
print(f'  RMSLE: {rmsle:.4f}')
print(f'  RMSE:  {rmse:.2f}')
print(f'  MAE:   {mae:.2f}')
print(f'  Prosjecna stvarna prodaja: {val_actual.mean():.2f}')
print(f'  MAE kao % prosjecne prodaje: {mae/val_actual.mean()*100:.1f}%')

In [ ]:
baseline_pred  = np.expm1(X_val['lag_7'])
baseline_rmsle = np.sqrt(np.mean((np.log1p(baseline_pred) - np.log1p(val_actual))**2))

print(f'Naive baseline (lag_7 kao predikcija):')
print(f'  RMSLE: {baseline_rmsle:.4f}')
print(f'LightGBM poboljsanje vs baseline: {(baseline_rmsle - rmsle)/baseline_rmsle*100:.1f}%')

## 5. Feature importance

In [ ]:
importance = pd.DataFrame({
    'feature':    model.feature_name(),
    'importance': model.feature_importance(importance_type='gain'),
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
importance.head(20).sort_values('importance').plot(
    kind='barh', x='feature', y='importance', ax=ax,
    color='steelblue', legend=False
)
ax.set_title('Top 20 features po importance (gain)')
ax.set_xlabel('Gain')
plt.tight_layout()
plt.show()

print(importance.head(20).to_string(index=False))

## 6. Predikcije vs stvarne vrijednosti

In [ ]:
val_results = val_df[['date', 'store_nbr', 'family', 'sales']].copy()
val_results['pred'] = val_pred

daily_actual = val_results.groupby('date')['sales'].sum()
daily_pred   = val_results.groupby('date')['pred'].sum()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily_actual.index, daily_actual.values, label='Stvarna prodaja', color='steelblue', linewidth=2)
ax.plot(daily_pred.index,   daily_pred.values,   label='Predikcija',     color='darkorange', linewidth=2, linestyle='--')
ax.set_title('Ukupna dnevna prodaja: stvarna vs predvidjena (val set)')
ax.set_ylabel('Sales')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def rmsle_score(actual, pred):
    return np.sqrt(np.mean((np.log1p(pred.clip(0)) - np.log1p(actual))**2))

rmsle_by_family = (
    val_results
    .groupby('family')
    .apply(lambda g: rmsle_score(g['sales'], g['pred']))
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={0: 'rmsle'})
)

fig, ax = plt.subplots(figsize=(12, 7))
rmsle_by_family.sort_values('rmsle').plot(
    kind='barh', x='family', y='rmsle', ax=ax,
    color='steelblue', legend=False
)
ax.set_title('RMSLE po kategoriji proizvoda')
ax.set_xlabel('RMSLE')
plt.tight_layout()
plt.show()

print('Najgore kategorije:')
print(rmsle_by_family.head(10).to_string(index=False))

## 7. Snimanje modela

In [ ]:
model_path = MODELS + 'lgbm_prophet.pkl'
with open(model_path, 'wb') as f:
    pickle.dump({
        'model':    model,
        'features': FEATURES,
        'val_rmsle': rmsle,
        'params':   params,
    }, f)

print(f'Model snimljen: {model_path}')
print(f'Best iteration: {model.best_iteration}')
print(f'Val RMSLE:      {rmsle:.4f}')